In [ ]:
"""
PIPELINE STAGE 5: Hallucination Candidate Extraction
----------------------------------------------------
Role in Thesis:
    Operationalizes clinically relevant mismatch patterns. Extracts observations 
    that appear *only* in the generated output (H_i = G_i \ R_i) as candidates 
    for unsupported findings (hallucinations).
    
Review Integration:
    Prepares a clean template for manual expert evaluation, labeling candidates 
    as 'supported', 'unsupported', or 'uncertain'. These labels act as the bridge 
    between automated extraction and probabilistic downstream modeling.
"""

In [ ]:
# =====================================================================
# PREPARING OBSERVATION-LEVEL REVIEW TEMPLATE
# =====================================================================
# Instead of scoring full reports, we extract the precise generated-only 
# observations (H_i) for targeted manual expert review. This reduces cognitive 
# load and links the automated extraction back to a human interpretive layer.

# =====================================================================
# REINTEGRATING MANUAL LABELS & DEFINING HALLUCINATION RATES
# =====================================================================
# Mocks/Integrates the evaluated manual labels ('supported', 'unsupported', 'uncertain').
# 
# Operational Definitions for Sensitivity Analysis:
# - STRICT Formulation: Counts only explicitly 'unsupported' claims as hallucinations.
# - LENIENT Formulation: Additionally counts 'uncertain' additions as hallucinations.
# These distinct operationalizations are passed to the downstream regression.

import os
import sys
import re

os.environ.setdefault("PYTENSOR_FLAGS", "cxx=,optimizer_excluding=fusion")

!{sys.executable} -m pip install -q pandas numpy

from pathlib import Path
import pandas as pd
import numpy as np
from IPython.display import display

# ------------------------------------------------------------------
# Paths
# ------------------------------------------------------------------
PROJECT_ROOT = Path.cwd().parent
CASE_DATAPATH = PROJECT_ROOT / "outputs" / "radgraph_analysis_1500_subset" / "case_level_comparison.csv"
RAD_DATAPATH = PROJECT_ROOT / "outputs" / "radgraph_extractions_1500_subset" / "combined_radgraph_processed.csv"
OUT = PROJECT_ROOT / "outputs" / "manual_review_1500_subset"
OUT.mkdir(parents=True, exist_ok=True)

print("Current working directory:", Path.cwd())
print("CASE_DATAPATH:", CASE_DATAPATH)
print("Resolved CASE_DATAPATH:", CASE_DATAPATH.resolve())
print("CASE_DATAPATH exists:", CASE_DATAPATH.exists())

print("RAD_DATAPATH:", RAD_DATAPATH)
print("Resolved RAD_DATAPATH:", RAD_DATAPATH.resolve())
print("RAD_DATAPATH exists:", RAD_DATAPATH.exists())

if not CASE_DATAPATH.exists():
    raise FileNotFoundError(f"Could not find {CASE_DATAPATH}")

if not RAD_DATAPATH.exists():
    raise FileNotFoundError(f"Could not find {RAD_DATAPATH}")

# ------------------------------------------------------------------
# Load data
# ------------------------------------------------------------------
case_df = pd.read_csv(CASE_DATAPATH)
rad_df = pd.read_csv(RAD_DATAPATH)

print("Loaded case_df shape:", case_df.shape)
print("Loaded rad_df shape:", rad_df.shape)

display(case_df.head())
display(rad_df.head())

# ------------------------------------------------------------------
# Normalize columns
# ------------------------------------------------------------------
case_rename_map = {
    "caseid": "case_id",
    "generatedreport": "generated_report",
    "referencereport": "reference_report",
    "generatedobservationcount": "generated_observation_count",
    "referenceobservationcount": "reference_observation_count",
    "sharedobservationcount": "shared_observation_count",
    "generatedonlycount": "generated_only_count",
    "referenceonlycount": "reference_only_count",
    "precisionobs": "precision_obs",
    "recallobs": "recall_obs",
    "f1obs": "f1_obs",
    "jaccardobs": "jaccard_obs",
    "sharedobservations": "shared_observations",
    "generatedonly": "generated_only",
    "referenceonly": "reference_only",
}

rad_rename_map = {
    "caseid": "case_id",
    "sourcetype": "source_type",
    "locatedat": "located_at",
    "suggestiveof": "suggestive_of",
}

case_df = case_df.rename(columns={c: case_rename_map.get(c, c) for c in case_df.columns})
rad_df = rad_df.rename(columns={c: rad_rename_map.get(c, c) for c in rad_df.columns})

# ------------------------------------------------------------------
# Type cleanup
# ------------------------------------------------------------------
case_df["case_id"] = pd.to_numeric(case_df["case_id"], errors="coerce").astype("Int64")
rad_df["case_id"] = pd.to_numeric(rad_df["case_id"], errors="coerce").astype("Int64")

numeric_cols = [
    "generated_observation_count",
    "reference_observation_count",
    "shared_observation_count",
    "generated_only_count",
    "reference_only_count",
    "precision_obs",
    "recall_obs",
    "f1_obs",
    "jaccard_obs",
]

for c in numeric_cols:
    if c in case_df.columns:
        case_df[c] = pd.to_numeric(case_df[c], errors="coerce")

text_cols_case = [
    "generated_report",
    "reference_report",
    "shared_observations",
    "generated_only",
    "reference_only",
]

for c in text_cols_case:
    if c in case_df.columns:
        case_df[c] = case_df[c].fillna("").astype(str)

text_cols_rad = ["source_type", "observation", "located_at", "suggestive_of", "tags"]

for c in text_cols_rad:
    if c in rad_df.columns:
        rad_df[c] = rad_df[c].fillna("").astype(str)

# ------------------------------------------------------------------
# Restrict RadGraph rows to case-level subset
# ------------------------------------------------------------------
case_ids = set(case_df["case_id"].dropna().astype(int).tolist())
rad_df = rad_df[rad_df["case_id"].isin(case_ids)].copy()

print("Filtered rad_df shape:", rad_df.shape)
print("Unique case_ids in case_df:", case_df["case_id"].nunique())
print("Unique case_ids in filtered rad_df:", rad_df["case_id"].nunique())

# ------------------------------------------------------------------
# Helpers
# ------------------------------------------------------------------
def norm_text(x):
    x = "" if pd.isna(x) else str(x).lower().strip()
    x = re.sub(r"\s+", " ", x)
    return x

# ------------------------------------------------------------------
# Build mismatch / generated-only candidate subset
# ------------------------------------------------------------------
rad_df["obs_norm"] = rad_df["observation"].map(norm_text)

ref_obs_by_case = (
    rad_df.loc[rad_df["source_type"].str.lower().eq("reference")]
    .groupby("case_id")["obs_norm"]
    .apply(lambda s: set(v for v in s if v))
    .to_dict()
)

gen_df = rad_df.loc[rad_df["source_type"].str.lower().eq("generated")].copy()

gen_df["is_generated_only_candidate"] = gen_df.apply(
    lambda r: r["obs_norm"] not in ref_obs_by_case.get(r["case_id"], set()),
    axis=1,
)

review_df = gen_df.loc[gen_df["is_generated_only_candidate"]].copy()

review_df = review_df.merge(
    case_df[
        [
            "case_id",
            "generated_report",
            "reference_report",
            "generated_only_count",
            "reference_only_count",
            "generated_only",
            "reference_only",
            "precision_obs",
            "recall_obs",
        ]
    ],
    on="case_id",
    how="left",
)

review_df = review_df.reset_index(drop=True)
review_df.insert(0, "review_item_id", [f"MR_{i+1:04d}" for i in range(len(review_df))])

# ------------------------------------------------------------------
# Add manual review fields
# ------------------------------------------------------------------
review_df["auto_subset"] = "generated_only_candidate"
review_df["manual_label"] = ""
review_df["reviewer"] = ""
review_df["review_date"] = ""
review_df["notes"] = ""
review_df["final_hallucination_strict"] = ""
review_df["final_hallucination_lenient"] = ""

full_cols = [
    "review_item_id",
    "case_id",
    "auto_subset",
    "observation",
    "located_at",
    "suggestive_of",
    "tags",
    "generated_only_count",
    "reference_only_count",
    "precision_obs",
    "recall_obs",
    "generated_report",
    "reference_report",
    "generated_only",
    "reference_only",
    "manual_label",
    "reviewer",
    "review_date",
    "notes",
    "final_hallucination_strict",
    "final_hallucination_lenient",
]

compact_cols = [
    "review_item_id",
    "case_id",
    "observation",
    "located_at",
    "suggestive_of",
    "tags",
    "manual_label",
    "reviewer",
    "review_date",
    "notes",
]

review_df = review_df[full_cols].copy()
review_compact_df = review_df[compact_cols].copy()

# ------------------------------------------------------------------
# Save manual review files
# ------------------------------------------------------------------
review_df.to_csv(OUT / "manual_review_template.csv", index=False)
review_compact_df.to_csv(OUT / "manual_review_template_compact.csv", index=False)

label_guide_df = pd.DataFrame(
    {
        "manual_label": ["unsupported", "supported", "uncertain"],
        "use_for_strict_hallucination": [1, 0, 0],
        "use_for_lenient_hallucination": [1, 0, 1],
        "definition": [
            "Generated finding not supported by image/reference context under review",
            "Generated finding is acceptable or supported",
            "Cannot be judged confidently or evidence is ambiguous",
        ],
    }
)
label_guide_df.to_csv(OUT / "manual_review_label_guide.csv", index=False)

# ------------------------------------------------------------------
# Diagnostics
# ------------------------------------------------------------------
alignment_df = (
    review_df.groupby("case_id")
    .size()
    .rename("derived_candidate_rows")
    .reset_index()
    .merge(
        case_df[["case_id", "generated_only_count"]],
        on="case_id",
        how="right",
    )
)

alignment_df["derived_candidate_rows"] = alignment_df["derived_candidate_rows"].fillna(0).astype(int)
alignment_df["generated_only_count"] = pd.to_numeric(
    alignment_df["generated_only_count"], errors="coerce"
).fillna(0).astype(int)
alignment_df["difference"] = alignment_df["derived_candidate_rows"] - alignment_df["generated_only_count"]

alignment_df.to_csv(OUT / "manual_review_alignment_check.csv", index=False)

case_summary_df = (
    review_df.groupby("case_id")
    .agg(
        n_candidates=("review_item_id", "count"),
        unique_observations=("observation", "nunique"),
    )
    .reset_index()
    .merge(
        case_df[["case_id", "generated_only_count", "precision_obs", "recall_obs"]],
        on="case_id",
        how="left",
    )
    .sort_values(["n_candidates", "generated_only_count"], ascending=[False, False])
)

case_summary_df.to_csv(OUT / "manual_review_case_summary.csv", index=False)

# ------------------------------------------------------------------
# Output preview
# ------------------------------------------------------------------
print("\nSaved outputs to:", OUT.resolve())
print("\nMain files:")
print("-", OUT / "manual_review_template.csv")
print("-", OUT / "manual_review_template_compact.csv")
print("-", OUT / "manual_review_label_guide.csv")
print("-", OUT / "manual_review_alignment_check.csv")
print("-", OUT / "manual_review_case_summary.csv")

print("\nReview rows:", len(review_df))
print("Review cases:", review_df["case_id"].nunique())
print("Exact alignment cases:", (alignment_df["difference"] == 0).sum())
print("Total cases:", len(alignment_df))

display(review_df.head(10))
display(case_summary_df.head(10))
display(alignment_df.head(10))
display(label_guide_df)


[notice] A new release of pip is available: 24.2 -> 26.0.1
[notice] To update, run: python -m pip install --upgrade pip
Current working directory: /workspace/thesis/notebooks
CASE_DATAPATH: /workspace/thesis/outputs/radgraph_analysis_1500_subset/case_level_comparison.csv
Resolved CASE_DATAPATH: /workspace/thesis/outputs/radgraph_analysis_1500_subset/case_level_comparison.csv
CASE_DATAPATH exists: True
RAD_DATAPATH: /workspace/thesis/outputs/radgraph_extractions_1500_subset/combined_radgraph_processed.csv
Resolved RAD_DATAPATH: /workspace/thesis/outputs/radgraph_extractions_1500_subset/combined_radgraph_processed.csv
RAD_DATAPATH exists: True
Loaded case_df shape: (272, 15)
Loaded rad_df shape: (2890, 6)


,case_id,generated_report,reference_report,generated_observation_count,reference_observation_count,shared_observation_count,generated_only_count,reference_only_count,precision_obs,recall_obs,f1_obs,jaccard_obs,shared_observations,generated_only,reference_only
0,1041064153850317,AP and lateral views of the chest. There is a ...,Large right-sided pleural effusion with underl...,5,4,2,3,2,0.400000,0.50,0.444444,0.285714,large effusion; small effusion,mild congestion; pneumothorax; unchanged,atelectasis; consolidation
1,1102224550126222,Patient is status post median sternotomy and a...,Slight improvement in mild pulmonary edema. Pa...,13,4,3,10,1,0.230769,0.75,0.352941,0.214286,atelectasis; infection; patchy opacities,acute abnormalities; aspiration; large effusio...,slight improvement mild edema
2,1156909359234239,The ET tube is 5 cm above the carina. The NG t...,1. ET tube terminating 5.1 cm above the carina...,8,4,2,6,2,0.250000,0.50,0.333333,0.200000,elevation; et tube,effusion; haze; increased infiltrate; ng tube ...,orogastric tube; worsening mild - to - moderat...
3,1160762850790949,The endotracheal tube is 4.5 cm above the cari...,1. Endotracheal tube appropriately retracted t...,8,4,2,6,2,0.250000,0.50,0.333333,0.200000,edema; endotracheal tube,catheter; effusion; enteric tube tip; moderate...,mild cardiomegaly; moderate greater effusions
4,1188092357045176,The lungs are clear. The cardiomediastinal sil...,Tiny right pleural effusion.,4,2,1,3,1,0.250000,0.50,0.333333,0.200000,effusion,clear; normal; pneumothorax,tiny


,case_id,source_type,observation,located_at,suggestive_of,tags
0,1004616657379357,generated,wire fractured,NaN,NaN,definitely present
1,1004616657379357,generated,normal,heart size,NaN,definitely present
2,1004616657379357,generated,unremarkable,mediastinal and hilar contours,NaN,definitely present
3,1004616657379357,generated,normal,pulmonary vasculature,NaN,definitely present
4,1004616657379357,generated,calcified granuloma,right lower lobe,NaN,definitely present


Filtered rad_df shape: (2890, 6)
Unique case_ids in case_df: 272
Unique case_ids in filtered rad_df: 271

Saved outputs to: /workspace/thesis/outputs/manual_review_1500_subset

Main files:
- /workspace/thesis/outputs/manual_review_1500_subset/manual_review_template.csv
- /workspace/thesis/outputs/manual_review_1500_subset/manual_review_template_compact.csv
- /workspace/thesis/outputs/manual_review_1500_subset/manual_review_label_guide.csv
- /workspace/thesis/outputs/manual_review_1500_subset/manual_review_alignment_check.csv
- /workspace/thesis/outputs/manual_review_1500_subset/manual_review_case_summary.csv

Review rows: 2015
Review cases: 271
Exact alignment cases: 242
Total cases: 272


,review_item_id,case_id,auto_subset,observation,located_at,suggestive_of,tags,generated_only_count,reference_only_count,precision_obs,...,generated_report,reference_report,generated_only,reference_only,manual_label,reviewer,review_date,notes,final_hallucination_strict,final_hallucination_lenient
0,MR_0001,1004616657379357,generated_only_candidate,wire fractured,,,definitely present,8,2,0.000000,...,The patient is status post median sternotomy a...,No radiographic findings to suggest pneumonia.,acute abnormalities; calcified granuloma; clea...,findings; pneumonia,,,,,,
1,MR_0002,1004616657379357,generated_only_candidate,normal,heart size,,definitely present,8,2,0.000000,...,The patient is status post median sternotomy a...,No radiographic findings to suggest pneumonia.,acute abnormalities; calcified granuloma; clea...,findings; pneumonia,,,,,,
2,MR_0003,1004616657379357,generated_only_candidate,unremarkable,mediastinal and hilar contours,,definitely present,8,2,0.000000,...,The patient is status post median sternotomy a...,No radiographic findings to suggest pneumonia.,acute abnormalities; calcified granuloma; clea...,findings; pneumonia,,,,,,
3,MR_0004,1004616657379357,generated_only_candidate,normal,pulmonary vasculature,,definitely present,8,2,0.000000,...,The patient is status post median sternotomy a...,No radiographic findings to suggest pneumonia.,acute abnormalities; calcified granuloma; clea...,findings; pneumonia,,,,,,
4,MR_0005,1004616657379357,generated_only_candidate,calcified granuloma,right lower lobe,,definitely present,8,2,0.000000,...,The patient is status post median sternotomy a...,No radiographic findings to suggest pneumonia.,acute abnormalities; calcified granuloma; clea...,findings; pneumonia,,,,,,
5,MR_0006,1004616657379357,generated_only_candidate,clear,,,definitely present,8,2,0.000000,...,The patient is status post median sternotomy a...,No radiographic findings to suggest pneumonia.,acute abnormalities; calcified granuloma; clea...,findings; pneumonia,,,,,,
6,MR_0007,1004616657379357,generated_only_candidate,effusion,pleural,,definitely absent,8,2,0.000000,...,The patient is status post median sternotomy a...,No radiographic findings to suggest pneumonia.,acute abnormalities; calcified granuloma; clea...,findings; pneumonia,,,,,,
7,MR_0008,1004616657379357,generated_only_candidate,pneumothorax,,,definitely absent,8,2,0.000000,...,The patient is status post median sternotomy a...,No radiographic findings to suggest pneumonia.,acute abnormalities; calcified granuloma; clea...,findings; pneumonia,,,,,,
8,MR_0009,1004616657379357,generated_only_candidate,acute abnormalities,osseous,,definitely absent,8,2,0.000000,...,The patient is status post median sternotomy a...,No radiographic findings to suggest pneumonia.,acute abnormalities; calcified granuloma; clea...,findings; pneumonia,,,,,,
9,MR_0010,1026887750239281,generated_only_candidate,tracheostomy tube,,,definitely present,9,5,0.181818,...,Single AP upright portable view of the chest w...,1. Left PICC tip appears to terminate in the d...,consolidation; enlargement; large effusion; li...,aeration; effusion; left picc; residual streak...,,,,,,


,case_id,n_candidates,unique_observations,generated_only_count,precision_obs,recall_obs
178,1154028358773579,13,13,13,0.000000,0.000000
1,1004616653492798,12,12,12,0.000000,0.000000
89,1095905453881360,12,12,12,0.000000,0.000000
139,1141323651503417,12,12,12,0.000000,0.000000
265,1243354150247294,12,12,12,0.000000,0.000000
68,1093360952624179,12,11,11,0.000000,0.000000
148,1141323653994053,12,11,11,0.083333,0.333333
248,1218577556614076,12,11,11,0.000000,0.000000
41,1075418454594848,12,10,10,0.000000,0.000000
169,1147406557174042,12,10,10,0.090909,0.333333


,case_id,derived_candidate_rows,generated_only_count,difference
0,1041064153850317,3,3,0
1,1102224550126222,10,10,0
2,1156909359234239,6,6,0
3,1160762850790949,6,6,0
4,1188092357045176,3,3,0
5,1105293557502393,6,5,1
6,1093360950205123,6,6,0
7,1095905450128467,4,4,0
8,1026887757765703,8,7,1
9,1189309153024166,11,11,0


,manual_label,use_for_strict_hallucination,use_for_lenient_hallucination,definition
0,unsupported,1,1,Generated finding not supported by image/refer...
1,supported,0,0,Generated finding is acceptable or supported
2,uncertain,0,1,Cannot be judged confidently or evidence is am...
